# 进益资本杯 2026 · 端到端赛道 · 唯一推理 Notebook

规则要求**有且仅有一个** Notebook，评判程序 `import` 本文件里的 `main` 并以
`main(datasources, start_date, end_date)` 调用。

本文件**不包含任何预处理、窗口或模型结构代码**——那些都在共享代码 `ptop` 里。
Notebook 只做三件事：接收平台给的数据源名与区间、调用共享入口、返回规定的三列。

- **历史起点**：规则只授权「评测区间前一年」的历史，`main` 用
  `submission.history_start_for(start_date)` 动态推算，**不写死年份**。
- **结果范围**：输出仍被限制在 `start_date` 至 `end_date`。
- **离线 / 时限**：评测环境无外网、不假设 GPU；推理逐日分区，控制在 3 小时内。
- **不读标签**：推理不读取标签，不联网，不重新训练。

In [ ]:
import sys
from pathlib import Path

# 随包提交的共享代码在 src/ 下；镜像内也可能已预装 ptop，二者取其一即可。
for _base in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (_base / "src" / "ptop").is_dir():
        _src = str(_base / "src")
        if _src not in sys.path:
            sys.path.insert(0, _src)
        break

import dai
import ptop.submission as submission

# 提交前替换为随包提交的权重目录（save_bundle 的产物）。
BUNDLE_PATH = "weights/epoch0007"

# 必须与训练时用的频率一致：窗口长度按「槽数」定义，换频率会静默改变模型看到的时间跨度。
TABLE_KEY = "bar30m"

# 评测环境无外网、不假设 GPU；逐日分批即可在 3 小时内跑完。
DEVICE = "cpu"
MICROBATCH_SIZE = 256


def main(datasources, start_date, end_date):
    """按逻辑数据源名取表；返回且仅返回 date, instrument, score 三列。

    历史起点不写死：规则只授权「评测区间前一年」的历史，按传入的 start_date 动态推算。
    结果仍被限制在 start_date 至 end_date，并由逐日分区推理控制内存与总时长。
    """
    return submission.run_submission(
        query=dai.query,
        datasources=datasources,
        bundle_path=BUNDLE_PATH,
        start_date=start_date,
        end_date=end_date,
        history_start=submission.history_start_for(start_date),
        table_key=TABLE_KEY,
        device=DEVICE,
        microbatch_size=MICROBATCH_SIZE,
    )

## 提交前自检

下面这段是**可选**的自检，用来在本地/演练环境提前暴露问题。正式评测只调用 `main`。
输出已经过官方审计：列顺序、分钟精度、唯一键、有限值、每时点非空且缺失率 ≤ 40%。

In [ ]:
# 演练时的用法（把 datasources 换成平台实际传入的映射）：
# frame = main({"bar30m": "cpt_jyc_2026_stock_bar30m"}, "2024-01-02", "2024-01-05")
# assert list(frame.columns) == ["date", "instrument", "score"]
# print(frame.head(), frame.shape)
